# Step-by-step demo of NEDAS using the vort2d model



## Setup

NEDAS is available from the PyPI platform. You can install with ```pip install NEDAS```.

You can also clone the `NEDAS` repository and install in editable mode ```cd NEDAS; pip install -e .``` for active code development.

In [ ]:
# In Google Colab, run this cell to install
# only need to do this once, you may need to restart the kernel after this
to_install = False

if to_install:
    # 1. Install the latest version of NEDAS on develop branch
    %cd ~
    %rm -rf NEDAS
    !git clone https://github.com/nansencenter/NEDAS.git
    %cd NEDAS
    %pip install .
    
    # 2. Install additional dependencies
    # numba provides JIT compilation of python function to speed it up during runtime
    %pip install numba
    # cmocean provides better colormaps for visualization
    %pip install cmocean
    # ipython widgets for interactive plots
    %pip install ipywidgets
    
    # 3. Clone the tutorial repo too
    %mkdir ~/work
    %cd ~/work
    %rm -rf NEDAS_tutorials
    !git clone https://github.com/myying/NEDAS_tutorials.git
    %cd ~/work/NEDAS_tutorials


Check how many CPUs are available on your system. You can then set `nproc`, `nproc_util` in the configuration accordingly

**Notes**:
- If `mpi4py` is not supported, you shall use `nproc=1`
- For the vort2d model with relatively small dimension, `nproc=1` is already fast and increasing it won't speed up due to additional overheads

In [ ]:
import os
os.cpu_count()

In [ ]:
# this shell command checks if mpi4py is working normally
import subprocess
subprocess.run("mpirun -n 4 python -c 'from mpi4py import MPI; print(MPI.COMM_WORLD.Get_rank())'", shell=True)

## Prerequisites

Load modules, initialize objects, define utility functions

In [ ]:
# utility modules
import os
import numpy as np
from datetime import datetime, timedelta, timezone

# key NEDAS modules
from NEDAS.config import Config
from NEDAS.schemes import get_scheme

# a few graphic modules
import base64
import matplotlib.pyplot as plt
import cmocean
from matplotlib import cm
from matplotlib.animation import FuncAnimation
import ipywidgets as widgets
from PIL import Image
from IPython.display import HTML, display, clear_output
from NEDAS.utils.graphics import add_colorbar, adjust_ax_size


### Configuration

A YAML file `vort2d/config.yml` contains all the settings for an experiment.

See [Configuration file](https://nedas.readthedocs.io/en/latest/config_file.html) documentation for more details.

In command line, you can run the experiment with `python -m NEDAS -c vort2d/config.yml`

Here we setup the `config` object in an interactive environment:

In [ ]:
# load configuration YAML file
# additional kwargs will overwrite the values in the file
config = Config(config_file="vort2d/config.yml", nproc=1, quiet=False)

# you can also change values like this
config.nens = 1000
#config.debug = False
#config.io_mode = 'online'
#config.cycle_period = 6
config.model_def['vort2d']['dt'] = 10
config.model_def['vort2d']['loc_sprd'] = 10_000
config.model_def['vort2d']['Vbg'] = 0

config.obs_def[0]['hroi'] = 150_000
config.obs_def[0]['nobs'] = 1
config.dataset_def['vort2d']['network_type'] = 'targeted'
config.dataset_def['vort2d']['obs_range'] = 50_000

config.inflation_def['adaptive'] = False

# to list all configuration variables
#config.__dict__

In [ ]:
# Once you are happy with the configuration,
# you can initialize the main Scheme object
scheme = get_scheme(config)

In [ ]:
# For convenience, we make alias of a few frequently used objects
from typing import cast

# model object, which is a Vort2DModel(Model) instance
from NEDAS.models.vort2d import Vort2DModel
model = cast(Vort2DModel, scheme.c.models['vort2d'])

# dataset object, which is a Vort2DObs(SyntheticObs) instance
from NEDAS.datasets.vort2d import Vort2DObs
dataset = cast(Vort2DObs, scheme.c.datasets['vort2d'])

# and the analysis grid (same as model grid), which is a RegularGrid instance
from NEDAS.grid import RegularGrid
grid = cast(RegularGrid, scheme.c.grid)


### Some diagnostic tools

To visualize the state of the vort2d model, it 

Diagnostics to measure the performance, in both physical and spectral spaces

- root-mean-square error (RMSE)
- ensemble spread

Visualization

- vorticity maps
- spectrum
- sawtooth time series

In [ ]:
from NEDAS.utils.spatial_operation import gradx, grady

# compute vorticity (1/s) from velocity (m/s)
def uv2zeta(vel):
    u, v = vel[0], vel[1]
    zeta = gradx(v, grid.dx, grid.cyclic_dim) - grady(u, grid.dy, grid.cyclic_dim)
    return zeta

# color map and range of vorticity to show (1/s)
vort_cmap = getattr(cmocean.cm, 'curl')
vort_min = -5e-3
vort_max = 5e-3
vort_intv = 1e-3

In [ ]:
# domain-averaged RMSE
def rmse(Xens, Xt):
    return np.sqrt(np.mean((np.mean(Xens, axis=1)-Xt)**2, axis=(1,2,3)))

# domain-averaged ensemble spread
def sprd(Xens):
    return np.sqrt(np.mean(np.std(Xens, axis=1)**2, axis=(1,2,3)))

# spectrum of error and ensemble spread
from NEDAS.diag.metrics.spectral import pwrspec2d
from NEDAS.utils.fft_lib import get_wn

def err_spec(Xens, Xt):
    wn, err_pwr = pwrspec2d(np.mean(Xens, axis=0)-Xt)
    return wn, err_pwr

def sprd_spec(Xens):
    nens, nv, nj, ni = Xens.shape
    ki, kj = get_wn(Xens[-2:])
    nup = int(max(ki.max(), kj.max()))
    Xmean = np.mean(Xens, axis=0)
    pwr_ens = np.zeros((nens, nv, nup))
    wn = None
    for m in range(nens):
        wn, pwr_ens[m, ...] = pwrspec2d(Xens[m, ...] - Xmean)
    sprd_pwr = np.sum(pwr_ens, axis=0) / (nens-1)
    return wn, sprd_pwr

In [ ]:
def set_map_axis(ax):
    cticks = np.array([5, 15, 25, 35, 45])
    ax.set_xticks(grid.x[0,cticks])
    ax.set_xticklabels(((grid.x[0,cticks] - grid.Lx/2)/1e3).astype(int))
    ax.set_yticks(grid.y[cticks,0])
    ax.set_yticklabels(((grid.y[cticks,0] - grid.Ly/2)/1e3).astype(int))
    ax.set_xlabel(r'$x$ (km)')
    ax.set_ylabel(r'$y$ (km)')

# true vorticity map, highlighted contour in black, and ensemble members in colors
def plot_vorticity_map(ax, n, ts, Xt, Xens):
    ax.clear()
    vort_truth = uv2zeta(Xt[n])
    vort_truth[np.where(vort_truth>vort_max)] = vort_max
    vort_truth[np.where(vort_truth<vort_min)] = vort_min
    ax.contourf(grid.x, grid.y, vort_truth, np.arange(vort_min, vort_max+vort_intv, vort_intv), cmap=vort_cmap)
    clvs = [-1e-3, 1e-3]
    cmap = [getattr(plt.cm, 'jet')(x) for x in np.linspace(0, 1, scheme.c.nens)]
    for m in range(scheme.c.nens):
        vort_mem = uv2zeta(Xens[n,m])
        ax.contour(grid.x, grid.y, vort_mem, clvs, colors=[cmap[m][0:3]], linewidths=1)
    ax.contour(grid.x, grid.y, vort_truth, clvs, colors='k', linewidths=2)
    ax.set_title(fr"vorticity $t$={ts[n]:02}h")
    set_map_axis(ax)

# power spectra of error and ensemble spread
def plot_spectrum(ax, n, ts, Xt, Xens):
    ax.clear()
    wn, pwr = pwrspec2d(Xt[n])
    ax.loglog(wn, np.mean(pwr, axis=0), color='k', linewidth=2, label='truth')
    wn, err_pwr = err_spec(Xens[n], Xt[n])
    ax.loglog(wn, np.mean(err_pwr, axis=0), color='g', linewidth=3, label='rmse')
    wn, sprd_pwr = sprd_spec(Xens[n])
    ax.loglog(wn, np.mean(sprd_pwr, axis=0), color='r', linestyle=':', linewidth=2, label='sprd')
    ax.legend()
    ax.set_title(fr'spectrum $t$={ts[n]:02}h')
    ax.set_xlim([0.8, 12])
    ax.set_ylim([1e-5, 1e2])
    lengths = np.array([500, 200, 100, 50, 20])
    ax.set_xticks(grid.Lx / 1e3 / lengths)
    ax.set_xticklabels(lengths)
    ax.set_xlabel('wavelength (km)')
    ax.set_ylabel(r'$m^2/s^2$')
    ax.grid()

# Sawtooth time series of error and ensemble spread
def plot_sawtooth_ts(ax, n, ts, rmse_ts, sprd_ts):
    dt = model.restart_dt
    ax.clear()
    ax.plot(ts[0:n+1], rmse_ts[0:n+1], color='g', linewidth=3)
    ax.plot(ts[0:n+1], sprd_ts[0:n+1], color='r', linestyle=':', linewidth=2)
    ax.plot(ts[n], rmse_ts[n], color='k', marker='+', markersize=10)
    ax.set_title('domain-avg rmse,sprd')
    ax.set_xlim([-dt, np.max(ts)+dt])
    ax.set_xticks([0, 12, 24, 36, 48])
    ax.set_ylim([0, 20])
    ax.set_xlabel(r'forecast $t$ (h)')
    ax.set_ylabel(r'$m/s$')
    ax.grid()

In [ ]:
def get_model_state(c, t, m):
    if t == c.time:
        try:
            state = c.io.call_method(c, 'post', model.read_var, name='velocity', member=m, time=t, model_src='vort2d')
        except:
            state = c.io.call_method(c, 'current', model.read_var, name='velocity', member=m, time=t, model_src='vort2d')
    else:
        try:
            state = c.io.call_method(c, 'prior', model.read_var, name='velocity', member=m, time=t, model_src='vort2d')
        except:
            state = c.io.call_method(c, 'current', model.read_var, name='velocity', member=m, time=t, model_src='vort2d')
    return state

### The verifying truth

We will run an Observing System Simulation Experiment (OSSE) for the vort2d model.

First step is to generate a "verifying truth", or a "nature run", which will be used both for generating synthetic observations and for verification of the results.

In [ ]:
# set time to the start of the experiment
scheme.c.time = config.time_start

# for the truth, we place the vortex in the center of the domain (without perturbing its position)
model.loc_sprd = 0

# run the model from time_start to time_end and save results
# in offline IO mode, the truth states are saved under model.truth_dir (vort2d/work/truth/*nc files)
scheme.run_step('prepare_truth')

In [ ]:
scheme.c.time = config.time_start
truth_state = []
times = []
while scheme.c.time < config.time_end:
    t = scheme.c.time
    while t <= scheme.c.next_time:
        times.append(t)
        truth = scheme.c.io.call_method(scheme.c, 'truth', model.read_var, name='velocity', member=None, time=t, model_src='vort2d')
        truth_state.append(truth)
        t += model.restart_dt * timedelta(hours=1)
    scheme.c.time = scheme.c.next_time

### Generation of initial ensemble

Second,

In [ ]:
scheme.c.time = config.time_start
model.loc_sprd = config.model_def['vort2d']['loc_sprd']
scheme.run_step('prepare_init_ensemble')

model.save_memory('current', time=scheme.c.time)

## Data assimilation


### Assimilating a single observation

In [ ]:
scheme.c.time = datetime(2001,1,1, tzinfo=timezone.utc)
scheme.run_step('preprocess')

%mkdir -p vort2d/work/cycle/200101010000/analysis

In [ ]:
from NEDAS.core import State, Obs

scheme.c.time = datetime(2001,1,1, tzinfo=timezone.utc)
scheme.c.progress.call_stack_max_level = None
scheme.c.progress.call_stack = []

scheme.c.progress.push('filter')

scheme.c.iter = 0
scheme.c.update_assim_tools()

scheme.c.state = State(scheme.c)
scheme.c.logger('Prepare prior state')(scheme.c.state.prepare_state)(scheme.c)

In [ ]:
scheme.c.obs = Obs(scheme.c)
scheme.c.logger('Prepare obs')(scheme.c.obs.prepare_obs)(scheme.c)
scheme.c.logger('Prepare obs from prior state')(scheme.c.obs.prepare_obs_from_state)(scheme.c, 'prior')

In [ ]:
from NEDAS.assim_tools.assimilators import get_assimilator

scheme.c.assimilator = get_assimilator(scheme.c)

scheme.c.logger('Assimilator')(scheme.c.assimilator.assimilate)(scheme.c)

In [ ]:
from NEDAS.assim_tools.updators import get_updator
scheme.c.updator = get_updator(scheme.c)
scheme.c.logger('Updator')(scheme.c.updator.update)(scheme.c)

In [ ]:
# compute correlation
fld_prior_ens = np.zeros((config.nens,)+grid.x.shape)
for m in range(config.nens):
    fld_prior_ens[m,...] = scheme.c.state.fields_prior[m,0][0,...]
fld_prior_mean = np.mean(fld_prior_ens, axis=0)

obs_prior_ens = np.array([scheme.c.obs.obs_prior[m,0][0] for m in range(config.nens)])
obs_prior_mean = np.mean(obs_prior_ens, axis=0)

cov = np.zeros(grid.x.shape)
fld_prior_var = np.zeros(grid.x.shape)
obs_prior_var = 0
for m in range(config.nens):
    cov += ((fld_prior_ens[m,...] - fld_prior_mean) * (obs_prior_ens[m] - obs_prior_mean))
    fld_prior_var += (fld_prior_ens[m,...] - fld_prior_mean)**2
    obs_prior_var += (obs_prior_ens[m] - obs_prior_mean)**2
cov /= config.nens-1
fld_prior_var /= config.nens-1
obs_prior_var /= config.nens-1

fld_prior_var[np.where(fld_prior_var==0)] = 1e-10

corr = cov / np.sqrt(fld_prior_var * obs_prior_var)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12,4), constrained_layout=True)
vmax = 30
state_i, state_j = 30, 30
state_x = grid.x[state_j, state_i]
state_y = grid.y[state_j, state_i]
state_prior = fld_prior_ens[:, state_j, state_i]

# truth field; obs; and state(i,j) location
i = times.index(scheme.c.time)
grid.plot_vectors(ax[0], truth_state[i], V=vmax, linecolor=[.7,.7,.7])                  
obs_rec_id = 0
obs_seq = scheme.c.obs.obs_seq[obs_rec_id]
obs_val = obs_seq['obs']
obs_x = obs_seq['x'] 
obs_y = obs_seq['y']
grid.plot_scatter(ax[0], obs_val, vmax=vmax, x=obs_x, y=obs_y, is_vector=True, linecolor='k', linewidth=1)
ax[0].plot(obs_x, obs_y, color='k', marker='o', markersize=3, zorder=10)
ax[0].plot(state_x, state_y, color='k', marker='+', markersize=10, zorder=10)
set_map_axis(ax[0])

# correlation map
cmap = getattr(cmocean.cm, 'balance') 
#grid.plot_field(ax[1], truth_state[i][0], -model.Vmax, model.Vmax, cmap)
#grid.plot_scatter(ax[1], obs_val[0], -model.Vmax, model.Vmax, 15, cmap, x=obs_x, y=obs_y)
grid.plot_field(ax[1], corr, vmin=-1, vmax=1, cmap=cmap)
#add_colorbar(fig, ax[1], cmap, -1, 1)
ax[1].plot(obs_x, obs_y, color='k', marker='o', markersize=3, zorder=10)
ax[1].plot(state_x, state_y, color='k', marker='+', markersize=10, zorder=10)
set_map_axis(ax[1])

# scatter plot of obs-state relation
ax[2].scatter(obs_prior_ens, state_prior)


## Running experiments

Running the entire scheme, cycling from time_start to time_end

A few things to play with

Since the step by step shown, only works with nproc=1

If you will try nproc>1, use scheme() directly

### Cases

- `freerun`: a free ensemble forecast (without DA)
- `assim_global`
- `assim_targeted`

In [ ]:
casename = 'freerun'

config = Config(config_file='vort2d/config.yml', run_analysis=False, quiet=True)

In [ ]:
# clear previous results
%rm -rf vort2d/work

model.memory = {}
dataset.memory = {}

## Run an experiment

Case

configuration

run scheme

check results, visualizations and performance diagnostics

In [ ]:
scheme.c.time = config.time_start
scheme.c.progress.call_stack = []
scheme.c.progress.call_stack_max_level = 2
scheme.c.config.quiet = False
scheme()

In [ ]:
scheme.c.progress.call_stack = []
scheme.c.progress.push('')
model.load_memory('current', scheme.c.time)

scheme.c.time = config.time_start
scheme.c.log_event("CYCLING START...")
while scheme.c.time < config.time_end:
    scheme.c.log_event(f"CURRENT CYCLE: {scheme.c.time} -> {scheme.c.next_time}")
    scheme.run_step('preprocess')
    scheme.run_step('postprocess')
    scheme.run_step('ensemble_forecast')
    scheme.c.time = scheme.c.next_time
scheme.c.log_event("CYCLING COMPLETE.", flag='finish')

In [ ]:
scheme.c.time = config.time_start
ens_state = []
while scheme.c.time < config.time_end:
    t = scheme.c.time
    while t <= scheme.c.next_time:
        ens = []
        for m in range(scheme.c.nens):
            ens.append(get_model_state(scheme.c, t, m))
        ens_state.append(ens)
        t += model.restart_dt * timedelta(hours=1)

    scheme.c.time = scheme.c.next_time

In [ ]:
nt = len(truth_state)
Xt = np.array(truth_state)
Xens = np.array(ens_state)
dt = model.restart_dt * timedelta(hours=1)
ts = [int((t - times[0]) / dt) for t in times]
rmse_ts = rmse(Xens, Xt)
sprd_ts = sprd(Xens)

%mkdir -p vort2d/work/plots

for n in range(nt):
    fig, ax = plt.subplots(1, 3, figsize=(9, 3), constrained_layout=True)
    plot_vorticity_map(ax[0], n, ts, Xt, Xens)
    plot_spectrum(ax[1], n, ts, Xt, Xens)
    plot_sawtooth_ts(ax[2], n, ts, rmse_ts, sprd_ts)
    plt.savefig(f"vort2d/work/plots/analysis_diag_{n+1:02}.png")
    plt.close()

In [ ]:
ns_out = [times.index(t) for t in np.unique(times)]


In [ ]:
Xt = np.array(truth_state)
Xens = np.array(ens_state)
dt = model.restart_dt * timedelta(hours=1)
ts = [int((t - times[0]) / dt) for t in times]
rmse_ts = rmse(Xens, Xt)
sprd_ts = sprd(Xens)

%mkdir -p vort2d/work/plots

for i in range(len(ns_out)):
    n = ns_out[i]
    fig, ax = plt.subplots(1, 3, figsize=(9, 3), constrained_layout=True)
    plot_vorticity_map(ax[0], n, ts, Xt, Xens)
    plot_spectrum(ax[1], n, ts, Xt, Xens)
    plot_sawtooth_ts(ax[2], n, ts, rmse_ts, sprd_ts)
    plt.savefig(f"vort2d/work/plots/freerun_diag_{i+1:02}.png")
    plt.close()

In [ ]:
frames = []
nt = len(truth_state)
for i in range(nt):
    path = f"vort2d/work/plots/analysis_diag_{i+1:02d}.png"
    frames.append(Image.open(path))

# Save as GIF
frames[-1].save('vort2d/analysis_diag_animation.gif',
               save_all=True,
               append_images=frames[0:],
               optimize=False,
               duration=200, # ms per frame
               loop=0)

# Display in notebook
display(HTML('<img src="vort2d/analysis_diag_animation.gif">'))

In [ ]:
display(Image(f"vort2d/analysis_diag_animation.gif"))

In [ ]:
nt = len(truth_state)

# 1. Create the Slider
slider = widgets.IntSlider(
    value=1, min=1, max=nt, 
    description='Frame:', 
    continuous_update=True,
    layout=widgets.Layout(width='500px')
)

# 2. Create the Output area
out = widgets.Output()

def update_frame(change):
    i = change['new']
    # Construct the path we know exists
    img_path = f"vort2d/work/plots/analysis_diag_{i:02d}.png"
    
    with out:
        clear_output(wait=True)
        try:
            # Read image and convert to Base64
            with open(img_path, "rb") as f:
                encoded = base64.b64encode(f.read()).decode("utf-8")
            
            # Display via HTML string (This bypasses the 404 path error)
            display(HTML(f'<img src="data:image/png;base64,{encoded}" style="width:100%; max-width:800px;">'))
        except Exception as e:
            print(f"Error loading frame {i}: {e}")

# 3. Link and Display
slider.observe(update_frame, names='value')

# Show initial frame
with out:
    update_frame({'new': 1})

display(widgets.VBox([slider, out]))